# Task 2 — Real Estate Housing Price Prediction (Ridge & Lasso Regression)
**Datheon Machine Learning & Data Science Internship**

| Property | Value |
|---|---|
| **Dataset** | [California Housing CSV](https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv) |
| **Models** | OLS · Ridge (L2) · Lasso (L1) |
| **Tuning** | GridSearchCV across 9 alpha values |
| **Metrics** | RMSE · MAE · R² |
| **Techniques** | Feature Engineering · Imputation · StandardScaler · One-Hot Encoding |


## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
SEED     = 42
DATA_URL = 'https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv'
COLORS   = {'OLS': '#4C72B0', 'Ridge': '#DD8452', 'Lasso': '#55A868'}
print('Libraries loaded ✔')

## 2. Load Dataset

In [ ]:
df = pd.read_csv(DATA_URL)
print(f'Shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum()>0]}')
df.head()

## 3. EDA — Correlation Heatmap & Target Distribution

In [ ]:
df_num = df.select_dtypes(include=[float, int])
target_col = 'median_house_value'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
corr = df_num.corr()
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), ax=axes[0],
            cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.4)
axes[0].set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')

axes[1].hist(df[target_col], bins=50, color='#4C72B0', edgecolor='white')
axes[1].axvline(df[target_col].mean(), color='red', linestyle='--', label=f'Mean: {df[target_col].mean():,.0f}')
axes[1].set_title('Target Distribution (Median House Value)', fontsize=13, fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.savefig('housing_1_eda.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
# Interaction features
df['rooms_per_household']     = df['total_rooms']    / df['households']
df['bedrooms_ratio']          = df['total_bedrooms'] / df['total_rooms']
df['population_per_household']= df['population']     / df['households']

# One-hot encode ocean_proximity
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=False)

FEATURES   = [c for c in df.columns if c != target_col]
X          = df[FEATURES].values
y          = df[target_col].values

# Impute missing
imputer    = SimpleImputer(strategy='median')
X          = imputer.fit_transform(X)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)

# Scale
scaler     = StandardScaler()
X_train_s  = scaler.fit_transform(X_train)
X_test_s   = scaler.transform(X_test)

print(f'Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}')
print(f'Engineered features ({len(FEATURES)}): {FEATURES[:6]} ...')

## 5. Model Training — OLS, Ridge, Lasso with GridSearchCV

In [ ]:
ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0, 1000.0]

models_config = {
    'OLS':   (LinearRegression(), {}),
    'Ridge': (Ridge(random_state=SEED),   {'alpha': ALPHAS}),
    'Lasso': (Lasso(max_iter=10000, random_state=SEED), {'alpha': ALPHAS}),
}

results    = {}
best_alpha = {}

for name, (estimator, param_grid) in models_config.items():
    if param_grid:
        gs = GridSearchCV(estimator, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
        gs.fit(X_train_s, y_train)
        model = gs.best_estimator_
        best_alpha[name] = gs.best_params_['alpha']
        print(f'{name} → best alpha={gs.best_params_["alpha"]}')
    else:
        model = estimator; model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'model': model, 'y_pred': y_pred,
                     'RMSE': rmse, 'MAE': mean_absolute_error(y_test, y_pred),
                     'R2': r2_score(y_test, y_pred), 'residuals': y_test - y_pred}
    print(f'  RMSE={rmse:,.1f}  MAE={results[name]["MAE"]:,.1f}  R²={results[name]["R2"]:.4f}')

## 6. Benchmark Report

In [ ]:
bench = pd.DataFrame([{'Model': n, 'RMSE': round(r['RMSE'],2), 'MAE': round(r['MAE'],2), 'R²': round(r['R2'],4)}
                       for n, r in results.items()]).set_index('Model')
bench

## 7. Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Predicted vs Actual — Housing Prices', fontsize=14, fontweight='bold')
for ax, (name, r) in zip(axes, results.items()):
    ax.scatter(r['y_pred'], y_test, alpha=0.3, s=10, color=COLORS[name])
    lims = [min(r['y_pred'].min(), y_test.min()), max(r['y_pred'].max(), y_test.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
    ax.set_title(f'{name}  |  R²={r["R2"]:.4f}', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.legend()
plt.tight_layout(); plt.savefig('housing_2_pred_vs_actual.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Residual Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Residual Distribution', fontsize=14, fontweight='bold')
for ax, (name, r) in zip(axes, results.items()):
    res = r['residuals']
    ax.hist(res, bins=60, color=COLORS[name], edgecolor='white', alpha=0.85)
    ax.axvline(0, color='red', linestyle='--', lw=1.5)
    ax.set_title(f'{name}  μ={res.mean():,.0f}  σ={res.std():,.0f}', fontsize=11)
plt.tight_layout(); plt.savefig('housing_3_residuals.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. Alpha Tuning Curves (Ridge & Lasso)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Regularization Alpha Tuning (5-Fold CV)', fontsize=14, fontweight='bold')
for ax, (reg_name, model_cls) in zip(axes, [('Ridge', Ridge), ('Lasso', Lasso)]):
    rmse_vals = []
    for a in ALPHAS:
        m = model_cls(alpha=a, max_iter=10000, random_state=SEED) if reg_name=='Lasso' else model_cls(alpha=a)
        cv = cross_val_score(m, X_train_s, y_train, cv=5, scoring='neg_root_mean_squared_error')
        rmse_vals.append(-cv.mean())
    ax.semilogx(ALPHAS, rmse_vals, marker='o', color=COLORS[reg_name], lw=2)
    best_a = best_alpha.get(reg_name, ALPHAS[np.argmin(rmse_vals)])
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best α={best_a}')
    ax.set_title(f'{reg_name} — RMSE vs Alpha', fontsize=12)
    ax.set_xlabel('Alpha (log scale)'); ax.set_ylabel('CV RMSE'); ax.legend()
plt.tight_layout(); plt.savefig('housing_4_alpha_tuning.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. Feature Coefficient Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 15 Feature Coefficients', fontsize=14, fontweight='bold')
for ax, reg_name in zip(axes, ['Ridge', 'Lasso']):
    coefs = pd.Series(results[reg_name]['model'].coef_, index=FEATURES)
    top   = coefs.abs().nlargest(15).index; coefs_top = coefs[top].sort_values()
    colors = ['#DD8452' if v>0 else '#4C72B0' for v in coefs_top]
    ax.barh(coefs_top.index, coefs_top.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{reg_name} Coefficients', fontsize=12)
plt.tight_layout(); plt.savefig('housing_5_coefficients.png', dpi=150, bbox_inches='tight'); plt.show()

## 11. Save Trained Housing Model to Pickle File
> Export best performing regression model bundle for deployment.


In [ ]:
import pickle, os
os.makedirs('../models', exist_ok=True)
housing_bundle = {
    'task': 'Task 2 — Real Estate Housing Price Prediction',
    'features': FEATURES,
    'model': results['Ridge']['model'],
    'scaler': scaler,
    'best_alpha': best_alpha.get('Ridge', 1.0)
}
with open('../models/housing_model.pkl', 'wb') as f:
    pickle.dump(housing_bundle, f)
print('✔ Housing model exported to ../models/housing_model.pkl')
